CLIP
https://github.com/openai/CLIP

In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import os
import clip
import torch
from torchvision import datasets, transforms
import numpy as np
import cv2
import skimage
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
%matplotlib inline

device = "cuda" if torch.cuda.is_available() else "cpu"

Dataset

In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip

In [ ]:
# !unzip /content/val2017.zip

Get CLIP model

In [ ]:
clip.available_models()

In [ ]:
model, preprocess = clip.load("RN50", device)
model.cuda().eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

In [ ]:
preprocess

Image search

In [ ]:
def encode_text(text):
    text = clip.tokenize([text]).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text)
    return text_features

def encode_image(image_path):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image)
    return image_features

In [ ]:
text_query = "mining farm"
text_features = encode_text(text_query)

In [ ]:
import random
image_folder_path = '/content/val2017'
text_query = '/content/val2017/' + random.choice(os.listdir(image_folder_path))
print(text_query)
text_features = encode_image(text_query)

In [ ]:
# import random
# random.choice(os.listdir(image_folder_path))

In [ ]:
image_folder_path = '/content/val2017'
image_features_list = []
image_paths = []

for image_path in tqdm(os.listdir(image_folder_path)):
    full_path = os.path.join(image_folder_path, image_path)
    image_features = encode_image(full_path)
    image_features_list.append(image_features)
    image_paths.append(full_path)

In [ ]:
image_features_list = torch.cat(image_features_list)

In [ ]:
similarity = torch.matmul(text_features, image_features_list.T).squeeze(0)

In [ ]:
top_k = 5
top_k_indices = similarity.topk(top_k).indices
top_images = [image_paths[i] for i in top_k_indices]

In [ ]:
top_images

In [ ]:
# img = cv2.imread("/content/val2017/000000570688.jpg")
# img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# plt.imshow(img)

In [ ]:
def plot_images(y_paths):
    fig = plt.figure(figsize=(20, 20), dpi=100)
    rows, cols = 1, 5
    for i, path in enumerate(y_paths[:rows*cols]):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (224, 224))

        fig.add_subplot(rows, cols, i+1)

        plt.imshow(img)
        plt.axis('off')

In [ ]:
plot_images(top_images)

In [ ]:
encode_text(text_query)

In [ ]:
encode_image(text_query)

Zero-shot classification

In [ ]:
# categories = ["black and white", "colour"]
categories = ["crowd", "animals", "food", "forest"]

In [ ]:
text_inputs = torch.cat([clip.tokenize(f"a photo of a {c}") for c in categories]).to(device)
# text_inputs = torch.cat([clip.tokenize(f"a {c} photo") for c in categories]).to(device)
with torch.no_grad():
    text_features = model.encode_text(text_inputs)

In [ ]:
result_images = []
result_titles = []
for image_path in os.listdir(image_folder_path)[:25]:
    full_path = os.path.join(image_folder_path, image_path)
    image_features = encode_image(full_path)
    similarity = (image_features @ text_features.T).softmax(dim=-1)
    predicted_category = categories[similarity.argmax()]
    result_titles.append(predicted_category + ' Prob:' + str(similarity[0][similarity.argmax().cpu().numpy()].cpu().numpy()))
    result_images.append(full_path)

In [ ]:
def plot_images(y_paths, y_titles):
    fig = plt.figure(figsize=(20, 20), dpi=100)
    rows, cols = 5, 5
    for i, path in enumerate(y_paths[:rows*cols]):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (224, 224))

        fig.add_subplot(rows, cols, i+1)
        plt.title(y_titles[i])
        plt.imshow(img)
        plt.axis('off')

In [ ]:
plot_images(result_images, result_titles)

BLIP

https://github.com/salesforce/LAVIS

In [ ]:
!pip install salesforce-lavis transformers tokenizers

In [ ]:
import lavis
from lavis.models import load_model_and_preprocess

In [ ]:
from lavis.models import model_zoo
print(model_zoo)

Image Captioning

In [ ]:
model, vis_processors, _ = load_model_and_preprocess(name="blip_caption", model_type="base_coco", is_eval=True, device=device)

In [ ]:
import random
image_folder_path = '/content/val2017'
# text_query = '/content/val2017/' + random.choice(os.listdir(image_folder_path))
# print(text_query)
queries = []
for i in range(10):
  text_query = '/content/val2017/' + random.choice(os.listdir(image_folder_path))
  queries.append(text_query)
  print(text_query)

In [ ]:
captions

In [ ]:
queries = ['/content/val2017/000000473974.jpg',
 '/content/val2017/000000024919.jpg',
 '/content/val2017/000000088250.jpg',
 '/content/val2017/000000263860.jpg',
 '/content/val2017/000000445365.jpg']

In [ ]:
captions = []
for path in queries:
    raw_img = Image.open(path).convert('RGB')
    image = vis_processors["eval"](raw_img).unsqueeze(0).to(device)
    captions.append(model.generate({"image": image})[0])

In [ ]:
def plot_images(y_paths, y_titles):
    fig = plt.figure(figsize=(20, 20), dpi=100)
    rows, cols = 3, 2
    for i, path in enumerate(y_paths[:rows*cols]):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))

        fig.add_subplot(rows, cols, i+1)
        plt.title(y_titles[i])
        plt.imshow(img)
        plt.axis('off')

In [ ]:
plot_images(queries, captions)

Visual QA

In [ ]:
model, vis_processors, txt_processors = load_model_and_preprocess(name="blip_vqa", model_type="vqav2", is_eval=True, device=device)

In [ ]:
question = "Which city is this photo taken?"

In [ ]:
path = '/content/vqa_1.jpg'
raw_img = Image.open(path).convert('RGB')

In [ ]:
plt.imshow(raw_img)

In [ ]:
image = vis_processors["eval"](raw_img).unsqueeze(0).to(device)

In [ ]:
question = txt_processors["eval"](question)

In [ ]:
model.predict_answers(samples={"image": image, "text_input": question}, inference_method="generate")

In [ ]:
path = '/content/vqa_2.jpg'
raw_img = Image.open(path).convert('RGB')

In [ ]:
plt.imshow(raw_img)

In [ ]:
question = "How many people are in this photo?"

In [ ]:
image = vis_processors["eval"](raw_img).unsqueeze(0).to(device)

In [ ]:
question = txt_processors["eval"](question)

In [ ]:
model.predict_answers(samples={"image": image, "text_input": question}, inference_method="generate")

In [ ]:
path = '/content/vqa_5.jpg'
raw_img = Image.open(path).convert('RGB')
plt.imshow(raw_img)

In [ ]:
question = "What is written on the image?"
image = vis_processors["eval"](raw_img).unsqueeze(0).to(device)
question = txt_processors["eval"](question)
model.predict_answers(samples={"image": image, "text_input": question}, inference_method="generate")

Feature extraction

In [ ]:
model, vis_processors, txt_processors = load_model_and_preprocess(name="blip2_feature_extractor", model_type="pretrain", is_eval=True, device=device)

In [ ]:
queries = ['/content/val2017/000000473974.jpg',
           '/content/val2017/000000024919.jpg',
           '/content/val2017/000000088250.jpg',
           '/content/val2017/000000263860.jpg',
           '/content/val2017/000000445365.jpg']

In [ ]:
captions

In [ ]:
raw_img = Image.open(queries[0]).convert('RGB')
plt.imshow(raw_img)

In [ ]:
image = vis_processors["eval"](raw_img).unsqueeze(0).to(device)
text_input = txt_processors["eval"](captions[0])
sample = {"image": image, "text_input": [text_input]}

In [ ]:
features_multimodal = model.extract_features(sample)
print(features_multimodal.multimodal_embeds.shape)

In [ ]:
features_multimodal

In [ ]:
features_image = model.extract_features(sample, mode="image")
features_text = model.extract_features(sample, mode="text")
print(features_image.image_embeds.shape, features_text.text_embeds.shape)

In [ ]:
features_image

In [ ]:
features_text

CLIP

In [ ]:
model, preprocess = clip.load("ViT-B/32", device)
model.cuda().eval()

In [ ]:
captions

In [ ]:
text_query = "a herd of zebra standing on top of a dry grass field"
text_features = encode_text(text_query)

In [ ]:
image_folder_path = '/content/val2017'
image_features_list = []
image_paths = []

for image_path in tqdm(os.listdir(image_folder_path)):
    full_path = os.path.join(image_folder_path, image_path)
    image_features = encode_image(full_path)
    image_features_list.append(image_features)
    image_paths.append(full_path)

In [ ]:
image_features_list = torch.cat(image_features_list)
similarity = torch.matmul(text_features, image_features_list.T).squeeze(0)

In [ ]:
top_k = 5
top_k_indices = similarity.topk(top_k).indices

top_images = [image_paths[i] for i in top_k_indices]

In [ ]:
top_images

In [ ]:
def plot_images(y_paths):
    fig = plt.figure(figsize=(20, 20), dpi=100)
    rows, cols = 1, 5
    for i, path in enumerate(y_paths[:rows*cols]):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (224, 224))

        fig.add_subplot(rows, cols, i+1)

        plt.imshow(img)
        plt.axis('off')

In [ ]:
plot_images(top_images)